# 요약 압축과 Compress

자동 요약 압축은 언제 실행되며 최근 메시지와 중요 조건을 어떻게 보존해야 할까요?


## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/03-요약-압축과-Compress.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.


## 패키지 버전 확인


In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.


langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 결정적 모델 준비


In [2]:
# 이 셀은 결정적 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
# 정해 둔 응답으로 미들웨어 실행 순서를 재현합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
class ScriptedModel(BaseChatModel):
    replies: list[AIMessage]
    index: int = 0
    @property
    def _llm_type(self) -> str: return "scripted-model"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        reply = self.replies[min(self.index, len(self.replies) - 1)]
        self.index += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "ScriptedModel":
        return self
@tool
def lookup_hours(day: str) -> str:
    """자료실 운영 시간을 요일로 조회합니다."""
    return "평일 09:00~20:00" if day == "평일" else "토요일 10:00~17:00"

## 긴 대화 준비

요약 압축은 오래된 메시지의 핵심을 짧은 메시지로 바꾸어 다음 모델 입력을 줄이는 처리입니다. `SummarizationMiddleware`는 트리거 조건을 넘으면 요약 모델을 호출하고, 최근 메시지 일부와 생성한 요약을 다음 요청에 전달합니다. 체크포인터의 과거 기록 전체를 삭제하는 기능과는 다릅니다. 무엇을 저장했는지와 이번 모델에 무엇을 보냈는지를 구분합니다.

실습의 `CaptureAll`과 `CaptureCompressed`는 모델이 실제로 받은 메시지를 `seen_before`, `seen_after`에 복사하는 관찰 장치입니다. 요약문의 길이만 줄었다고 성공으로 판단하지 않고, 대출 권수·기간·연장 조건·출처가 남았는지 문자열을 읽습니다. 압축은 손실을 허용하는 변환이므로 업무에 필요한 사실을 먼저 정해야 합니다.


In [3]:
# 이 셀은 긴 대화 준비 작업을 수행하고 확인할 값을 출력합니다.
from langchain_core.messages import HumanMessage, AIMessage
history = [
 HumanMessage("대출은 몇 권인가요?"), AIMessage("5권입니다 [E01 p.3]"),
 HumanMessage("기간은요?"), AIMessage("14일이며 예약이 없으면 7일 연장됩니다 [E01 p.3]"),
 HumanMessage("출처를 기억해 주세요."), HumanMessage("지금까지 조건을 정리해 주세요."),
]
print("원본 메시지 수:", len(history))

원본 메시지 수: 6


## Before: 원문 전체 전달


In [4]:
# 이 셀은 Before: 원문 전체 전달 작업을 수행하고 확인할 값을 출력합니다.
seen_before=[]
class CaptureAll(AgentMiddleware):
    def wrap_model_call(self, request: Any, handler: Any) -> Any:
        seen_before.extend(message.text for message in request.messages)
        return handler(request)
before_agent=create_agent(ScriptedModel(replies=[AIMessage("답")]),tools=[],middleware=[CaptureAll()])
before_agent.invoke({"messages":history})
print("모델 입력 메시지 수:",len(seen_before))
assert len(seen_before)==6

모델 입력 메시지 수: 6


## After: 자동 요약 적용


In [5]:
# 이 셀은 After: 자동 요약 적용 작업을 수행하고 확인할 값을 출력합니다.
from langchain.agents.middleware import SummarizationMiddleware
summary_model=ScriptedModel(replies=[AIMessage("대출은 5권·14일, 예약이 없으면 7일 연장 [E01 p.3]")])
seen_after=[]
class CaptureCompressed(AgentMiddleware):
    def wrap_model_call(self, request: Any, handler: Any) -> Any:
        seen_after.extend(message.text for message in request.messages)
        return handler(request)
after_agent=create_agent(
 ScriptedModel(replies=[AIMessage("압축된 조건을 확인했습니다.")]), tools=[],
 middleware=[SummarizationMiddleware(model=summary_model,trigger=("messages",4),keep=("messages",2)),CaptureCompressed()])
after_agent.invoke({"messages":history})
print("모델 입력:",seen_after)
joined=" ".join(seen_after)
assert "14일" in joined and "E01 p.3" in joined

모델 입력: ['Here is a summary of the conversation to date:\n\n대출은 5권·14일, 예약이 없으면 7일 연장 [E01 p.3]', '출처를 기억해 주세요.', '지금까지 조건을 정리해 주세요.']


## 실제 모델로 압축문 생성


In [6]:
# 이 셀은 실제 모델로 압축문 생성 작업을 수행하고 확인할 값을 출력합니다.
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file(): load_dotenv(folder / ".env"); break
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
live = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none")
prompt = "다음 기록을 한 문장으로 요약하되 권수, 기간, 연장 조건, 출처를 모두 보존하세요: " + " ".join(m.text for m in history[:4])
live_summary = live.invoke(prompt)
usage = live_summary.usage_metadata or {}
cost = usage.get("input_tokens",0)/1_000_000*0.75 + usage.get("output_tokens",0)/1_000_000*4.50
print("summary:", live_summary.text)
print("usage:", usage); print("estimated_cost_usd:", round(cost,6))
assert cost < 1.0 and live_summary.text.strip()

summary: 대출은 **5권**이며 **기간은 14일**이고 **예약이 없으면 7일 연장**되며, 출처는 **E01 p.3**입니다.
usage: {'input_tokens': 79, 'output_tokens': 45, 'total_tokens': 124, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
estimated_cost_usd: 0.000262


## 압축 전후 비교

친근한 말투나 반복된 인사는 요약해도 영향이 작지만, 승인 여부·계약 숫자·출처 식별자는 빠지면 후속 답이 달라질 수 있습니다. 이런 값은 상태의 별도 필드에 구조화해 보존하고 대화 요약에는 맥락만 맡기는 설계가 안전합니다.

메시지 수 감소와 토큰 수 감소는 관련되지만 같은 값은 아닙니다. 긴 한 메시지가 짧은 여러 메시지보다 토큰이 많을 수 있습니다. 이번 실습은 전달 메시지와 보존 문자열을 확인하며, 실제 모델별 토큰 절감률이나 요약 정확도 전체를 보장하지 않습니다.


In [7]:
# 이 셀은 압축 전후 비교 작업을 수행하고 확인할 값을 출력합니다.
print("before_count:",len(seen_before))
print("after_count:",len(seen_after))
print("estimated_cost_usd: 0.0 (결정적 로컬 모델)")

before_count: 6
after_count: 3
estimated_cost_usd: 0.0 (결정적 로컬 모델)
